# MFLib + MeasurementFramework release test

Run this notebook on FABRIC JupyterHub against every **release candidate** and every **final release** of mflib
or MeasurementFramework (MF). It builds a fresh slice the way a user would and exercises the main MF services end to end:

| Step | What is checked |
|---|---|
| 1. Install | the mflib under test is installed and imported (not some other copy) |
| 2. Slice | 3 Rocky 9 experiment nodes, each on its own FABNetv4 network, plus the MFLib meas node (default image) |
| 4. `MFLib(slice)` | the meas node cloned the expected MF release (tag, commit and `VERSION` file) |
| 5. `instrumentize(["prometheus", "route_flow"])` | services report success, per-route counters run on every node |
| 6-7. Traffic | fixed-rate iperf3 per route; Prometheus rates match; Grafana `mflib-route-flow` dashboard renders; `mf.info()` |
| 8. `instrumentize(["elk"])` | Elasticsearch/Kibana up, Beats on every node at the same version, documents from every node |
| 9. Summary | PASS/FAIL table, saved to `release_test_results-<mflib>-<mf>.json` |

**Which versions get tested.** When this notebook comes from the *MFLib Tests* artifact, the release tool put a
`requirements.txt` (pinned mflib version; TestPyPI for rcs) and `RELEASE_INFO.json` next to it, and the notebook uses
them. MF is whatever mflib clones at run time: by default the release GitHub labels **Latest**, which is what users get.

- **mflib rc / final:** run as-is.
- **MF rc:** set `MF_REF = "vA.B.CrcN"` (rc tags are never Latest).
- **MF final:** after `release.py final mf` has published the GitHub Release, set `EXPECT_MF = "vA.B.C"`. This proves
  that Latest really moved, so every installed mflib now gets the new MF.

Each run takes about 45-60 minutes, most of it slice creation and `MFLib(slice)`.

## 0. Settings

In [ ]:
import json, os, re

# --- mflib under test -------------------------------------------------------------------------------
# Taken from RELEASE_INFO.json / requirements.txt when they are next to this notebook (artifact tarball).
# Otherwise set MFLIB_VERSION (rc versions install from TestPyPI, finals from PyPI), or MFLIB_GIT_REF
# to test an unreleased branch, e.g. "nov26updates".
MFLIB_VERSION = None
MFLIB_GIT_REF = None

# --- MeasurementFramework under test -----------------------------------------------------------------
MF_REF = "latest"      # "latest" = the GitHub Latest release (mflib's default). Or a tag/branch, e.g. "v1.1.0rc1".
EXPECT_MF = None       # e.g. "v1.1.0": fail unless MF_REF resolves to exactly this tag (use for an MF final).

# --- slice ----------------------------------------------------------------------------------------------
FRESH_SLICE = True     # delete and rebuild the slice; keep True for release sign-off
RUN_ELK = True
DELETE_SLICE_AT_END = False
NODES = ["Node1", "Node2", "Node3"]
EXP_IMAGE = "default_rocky_9"
EXPECTED_MEAS_IMAGE = "default_ubuntu_24"   # MFLib.addMeasNode() default
# Sites to skip (no PTP, frequent problems). Review this list now and then.
AVOID = ["DALL", "GPN", "LBNL", "RENC", "SALT", "TACC", "UKY", "WASH", "NCSA", "LOSA", "GATECH", "INDI",
         "MAX", "MASS", "NEWY", "SRI", "UCSD", "FIU", "EDC", "CIEN"]
# UDP rate per route, each direction, in Mbit/s
ROUTE_RATES = {("Node1", "Node2"): 5, ("Node2", "Node3"): 10, ("Node1", "Node3"): 20}

# --- derived ------------------------------------------------------------------------------------------------
RELEASE_INFO = json.load(open("RELEASE_INFO.json")) if os.path.exists("RELEASE_INFO.json") else {}
if RELEASE_INFO and not (MFLIB_VERSION or MFLIB_GIT_REF):
    MFLIB_VERSION = RELEASE_INFO["mflib_version"]
assert MFLIB_VERSION or MFLIB_GIT_REF, "set MFLIB_VERSION or MFLIB_GIT_REF (no RELEASE_INFO.json here)"
label = MFLIB_VERSION or MFLIB_GIT_REF
SLICE_NAME = "mf-reltest-" + re.sub(r"[^A-Za-z0-9]+", "-", label).strip("-")
print("mflib under test:", label, "| MF ref:", MF_REF, "| slice:", SLICE_NAME)
if RELEASE_INFO:
    print("RELEASE_INFO.json:", json.dumps(RELEASE_INFO, indent=1))

In [ ]:
# PASS/FAIL bookkeeping used by every step. check() records and keeps going; require() stops the notebook
# when later steps can't run without it.
import datetime
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "ok": bool(ok), "detail": str(detail)})
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f": {detail}" if detail else ""))
    return bool(ok)
def require(name, ok, detail=""):
    if not check(name, ok, detail):
        raise RuntimeError(f"required check failed: {name} ({detail})")
STARTED = datetime.datetime.now(datetime.timezone.utc)

## 1. Install mflib
Installs into the user site. If mflib was already imported in this kernel, **restart the kernel and run from the top**
afterwards (the next cell tells you).

In [ ]:
import sys, subprocess
already_imported = "mflib" in sys.modules
if MFLIB_GIT_REF:
    # flit builds mflib by importing it, so build without isolation
    %pip install --user -q flit_core
    spec = f"git+https://github.com/fabric-testbed/mflib.git@{MFLIB_GIT_REF}"
    %pip install --user -q --force-reinstall --no-deps --no-build-isolation {spec}
else:
    if os.path.exists("requirements.txt"):
        req = open("requirements.txt").read()
        assert f"=={MFLIB_VERSION}" in req, f"requirements.txt does not pin {MFLIB_VERSION}:\n{req}"
    else:
        lines = (["--extra-index-url https://test.pypi.org/simple/"] if "rc" in MFLIB_VERSION else []) \
                + [f"fabrictestbed-mflib=={MFLIB_VERSION}"]
        open("requirements.txt", "w").write("\n".join(lines) + "\n")
    # replace whatever mflib is installed, then add any missing dependencies
    %pip install --user -q --force-reinstall --no-deps -r requirements.txt
    %pip install --user -q -r requirements.txt
out = subprocess.run([sys.executable, "-c", "import mflib; print(mflib.__version__); print(mflib.__file__)"],
                     capture_output=True, text=True)
print(out.stdout or out.stderr)
if already_imported:
    print(">>> mflib was already imported in this kernel: RESTART THE KERNEL and run from the top <<<")

In [ ]:
import time, tempfile, urllib.parse, importlib.metadata as md
import mflib
from mflib.mflib import MFLib
from mflib import route_flow
from mflib.mf_repo import resolve_mf_ref, latest_mf_release

print("mflib", mflib.__version__, "from", os.path.dirname(mflib.__file__))
if MFLIB_VERSION:
    require("mflib version", mflib.__version__ == MFLIB_VERSION == md.version("fabrictestbed-mflib"),
            f"imported {mflib.__version__}, installed {md.version('fabrictestbed-mflib')}, expected {MFLIB_VERSION}"
            " (restart the kernel if the install just ran)")
require("mflib has route_flow", hasattr(route_flow, "collect_experiment_topology"))

# Which MF the meas node will clone, resolved the same way mflib does it
MF_TAG = resolve_mf_ref(MF_REF, verbose=False)
print("MF_REF", MF_REF, "->", MF_TAG, "| GitHub Latest MF release:", latest_mf_release())
if MF_REF == "latest":
    require("GitHub Latest MF release found", MF_TAG != "main", "lookup failed, mflib would fall back to main")
if EXPECT_MF:
    require("MF resolves to the expected release", MF_TAG == EXPECT_MF, f"{MF_TAG} (expected {EXPECT_MF})")

## 2. Slice

In [ ]:
%%time
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()
fablib.verify_and_configure(validate_only=True)   # only reports the bastion key state; never creates a key

def slice_names():
    return [s.get_name() for s in fablib.get_slices()]

if SLICE_NAME in slice_names() and FRESH_SLICE:
    print(f"Deleting {SLICE_NAME} to rebuild it")
    fablib.get_slice(SLICE_NAME).delete()
    for _ in range(60):
        if SLICE_NAME not in slice_names():
            break
        time.sleep(10)
if SLICE_NAME in slice_names():
    print(f"Reusing existing slice {SLICE_NAME} (FRESH_SLICE=False): not a full release test")
else:
    sites = fablib.get_random_sites(count=len(NODES), avoid=AVOID)
    print("Sites:", sites)
    slice = fablib.new_slice(name=SLICE_NAME)
    for i, (name, site) in enumerate(zip(NODES, sites), start=1):
        node = slice.add_node(name=name, site=site, cores=2, ram=8, disk=10, image=EXP_IMAGE)
        iface = node.add_component(model="NIC_Basic", name=f"nic{i}").get_interfaces()[0]
        slice.add_l3network(name=f"net{i}", interfaces=[iface], type="IPv4")
    MFLib.addMeasNode(slice, disk=100)   # default meas node image
    slice.submit()

slice = fablib.get_slice(SLICE_NAME)
meas = slice.get_node(name="meas-node")
require("slice is StableOK", slice.get_state() == "StableOK", f"{slice.get_state()}, lease end {slice.get_lease_end()}")
check("meas node uses the default image", meas.get_image() == EXPECTED_MEAS_IMAGE, meas.get_image())

## 3. Experiment network addresses
route_flow needs the experiment IPs before `instrumentize`: the client collects them from the slice and sends them
to the meas node.

In [ ]:
from ipaddress import ip_address
EXP = {}
for i, name in enumerate(NODES, start=1):
    node, net = slice.get_node(name=name), slice.get_network(name=f"net{i}")
    iface = node.get_interface(network_name=f"net{i}")
    ip = iface.get_ip_addr()
    # get_ip_addr() returns the IPv6 link-local address until an IPv4 one is set
    if not ip or ip_address(str(ip)) not in net.get_subnet():
        ip = net.get_available_ips()[0]
        iface.ip_link_up()
        iface.ip_addr_add(addr=ip, subnet=net.get_subnet())
    EXP[name] = {"dev": iface.get_device_name(), "ip": str(ip), "site": node.get_site()}
    print(f"{name} {node.get_site():6} {iface.get_device_name()} {ip}")

# Route each peer's experiment subnet through this node's experiment gateway. A per-peer subnet is more specific
# than the 10.128.0.0/10 route fablib adds for the meas network, so experiment traffic uses the experiment NICs in
# both directions (with only the /10, replies leave through the meas NIC and iperf3 stalls).
for i, name in enumerate(NODES, start=1):
    node, gw = slice.get_node(name=name), slice.get_network(name=f"net{i}").get_gateway()
    for j, peer in enumerate(NODES, start=1):
        if peer != name:
            node.execute(f"sudo ip route replace {slice.get_network(name=f'net{j}').get_subnet()} "
                         f"via {gw} dev {EXP[name]['dev']}", quiet=True)
time.sleep(10)
for src in NODES:
    for dst in NODES:
        if src != dst:
            out, _ = slice.get_node(name=src).execute(f"ping -c 3 -W 2 {EXP[dst]['ip']} | grep -oE '[0-9]+% packet loss'",
                                                      quiet=True)
            check(f"ping {src} -> {dst}", out.strip() and not out.startswith("100%"), out.strip() or "no reply")

print(json.dumps(route_flow.collect_experiment_topology(slice), indent=1))   # what the client sends to the meas node

## 4. `MFLib(slice)`: set up the meas node
With `MF_REF = "latest"` this calls `MFLib(slice)` with no MF argument, exactly as users do.

In [ ]:
%%time
mf = MFLib(slice) if MF_REF == "latest" else MFLib(slice, mf_repo_branch=MF_REF)

In [ ]:
# The meas node must have cloned exactly MF_TAG. Run as mfuser: /home/mfuser is not readable by the ubuntu user.
out, _ = meas.execute("sudo -u mfuser bash -c 'cd /home/mfuser/mf_git && echo COMMIT=$(git rev-parse HEAD); "
                      "echo VERSION=$(cat VERSION 2>/dev/null); git log -1 --format=\"LOG=%h %cs %s\"; "
                      "echo SERVICES=$(ls /home/mfuser/services | tr \"\\n\" \" \")'", quiet=True)
print(out)
node_info = dict(l.split("=", 1) for l in out.splitlines() if "=" in l)
MF_COMMIT = node_info.get("COMMIT", "")

# Commit of MF_TAG on GitHub: branch head, lightweight tag, or (preferred) the commit an annotated tag points to
ls = subprocess.run(["git", "ls-remote", "https://github.com/fabric-testbed/MeasurementFramework.git",
                     f"refs/heads/{MF_TAG}", f"refs/tags/{MF_TAG}", f"refs/tags/{MF_TAG}^{{}}"],
                    capture_output=True, text=True).stdout.split()
remote = dict(zip(ls[1::2], ls[0::2]))
github_commit = remote.get(f"refs/tags/{MF_TAG}^{{}}") or remote.get(f"refs/tags/{MF_TAG}") or remote.get(f"refs/heads/{MF_TAG}", "?")
check("meas node cloned the expected MF", MF_COMMIT == github_commit,
      f"{MF_TAG}: node {MF_COMMIT[:7]}, GitHub {github_commit[:7]}")
if m := re.fullmatch(r"v(\d+)\.(\d+)\.(\d+)(rc\d+)?", MF_TAG):
    version = node_info.get("VERSION", "")
    # MF releases made with release/release.py (1.1.0 on) carry a VERSION file; older tags don't
    if version or tuple(map(int, m.groups()[:3])) >= (1, 1, 0):
        check("MF VERSION file matches the tag", version == MF_TAG[1:], f"VERSION={version or '(missing)'}")
check("meas node has the route_flow service", "route_flow" in node_info.get("SERVICES", ""), node_info.get("SERVICES"))

## 5. Instrumentize: Prometheus + route_flow

In [ ]:
%%time
results = mf.instrumentize(["prometheus", "route_flow"])
for key in ("prometheues", "grafana_manager", "route_flow"):   # (sic) mflib's key for Prometheus
    r = results.get(key) or {}
    check(f"instrumentize {key}", r.get("success"), r.get("msg", "") if not r.get("success") else "")
rf = results.get("route_flow") or {}
print(json.dumps({k: v for k, v in rf.items() if k != "topology"}, indent=1, default=str)[:3000])

In [ ]:
# Per-route counters on the experiment nodes (installed by route_flow as a systemd service)
for n in NODES:
    out, _ = slice.get_node(name=n).execute(
        "echo $(systemctl is-active mf-route-acct) $(systemctl is-enabled mf-route-acct) "
        "$(grep -c '^mf_route_bytes_total' /var/lib/node_exporter/mf_route.prom 2>/dev/null || echo 0)", quiet=True)
    active, enabled, series = (out.split() + ["?"] * 3)[:3]
    check(f"{n} mf-route-acct running", active == "active" and enabled == "enabled", f"{active}/{enabled}")
    check(f"{n} route counters exported", series.isdigit() and int(series) > 0, f"{series} series")

## 6. Traffic: fixed-rate iperf3 on each route
This is the experiment's load, not part of MF. Each pair runs bidirectional UDP at the rate in `ROUTE_RATES`.

In [ ]:
for n in NODES:
    slice.get_node(name=n).execute("sudo dnf install -y -q iperf3 > /dev/null 2>&1; "
                                   "pkill -f '[m]f_iperf_loop'; pkill iperf3; true", quiet=True)
for port, ((client, server), rate) in enumerate(ROUTE_RATES.items(), start=5201):
    ip = EXP[server]["ip"]
    slice.get_node(name=server).execute(f"iperf3 -s -D -p {port}", quiet=True)
    loop = f"while true; do iperf3 -c {ip} -p {port} -u -b {rate}M --bidir -t 0; sleep 2; done"
    slice.get_node(name=client).execute(
        f"echo '{loop}' > /tmp/mf_iperf_loop_{port}.sh; "
        f"setsid nohup bash /tmp/mf_iperf_loop_{port}.sh > /tmp/mf_iperf_{port}.log 2>&1 < /dev/null &", quiet=True)
    print(f"{client} <-> {server} ({ip}:{port}): UDP {rate} Mbit/s each way")

## 7. Prometheus, Grafana and `mf.info()`

In [ ]:
%%time
def prom(query):
    url = "http://localhost:9090/api/v1/query?query=" + urllib.parse.quote(query)
    out, _ = meas.execute(f"sudo docker exec fabric_prometheus-prometheus wget -qO- '{url}'", quiet=True)
    return json.loads(out)["data"]["result"] if out.strip() else []

def route_rates():
    rows = prom('rate(mf_route_bytes_total{direction="tx"}[1m])*8/1e6')
    got = {(r["metric"].get("mf_node"), r["metric"].get("mf_peer")): float(r["value"][1]) for r in rows}
    want = {(s, d): rate for (a, b), rate in ROUTE_RATES.items() for s, d in ((a, b), (b, a))}
    # counters include IP/UDP headers, so allow a little above the iperf3 rate
    return {k: (t, got.get(k), got.get(k) is not None and 0.9 * t <= got[k] <= 1.2 * t) for k, t in want.items()}

time.sleep(60)                 # rate() needs a minute of samples
for _ in range(12):            # then up to 3 more minutes for every route to settle
    rates = route_rates()
    if all(ok for _, _, ok in rates.values()):
        break
    time.sleep(15)
print(len(prom("mf_route_bytes_total")), "mf_route_bytes_total series in Prometheus")
for (src, dst), (target, got, ok) in rates.items():
    check(f"Prometheus rate {src} -> {dst}", ok, f"target {target} Mbit/s, measured {None if got is None else round(got, 2)}")

In [ ]:
# Grafana: the route_flow dashboard exists, and its topology panel renders (PNG saved next to this notebook)
script = r"""K=$(sudo sed -n 's/^[[:space:]]*apiKey[[:space:]]*[=:-][[:space:]]*//p' /home/mfuser/services/grafana_manager/configFile.txt | tr -d '[:space:]')
echo DASH=$(sudo curl -sk -H "Authorization: Bearer $K" https://localhost/grafana/api/dashboards/uid/mflib-route-flow | python3 -c 'import json,sys; d=json.load(sys.stdin); print(d["dashboard"]["title"], "|", len(d["dashboard"]["panels"]), "panels")' 2>/dev/null)
echo RENDER=$(sudo curl -sk -o /tmp/route_flow.png -w '%{http_code}' -H "Authorization: Bearer $K" 'https://localhost/grafana/render/d-solo/mflib-route-flow/x?orgId=1&panelId=1&width=1100&height=760&theme=dark')
"""
with tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False, newline="\n") as f:
    f.write(script)
meas.upload_file(f.name, "/tmp/route_flow_render.sh")
out, _ = meas.execute("bash /tmp/route_flow_render.sh", quiet=True)
g = dict(l.split("=", 1) for l in out.splitlines() if "=" in l)
check("Grafana mflib-route-flow dashboard", g.get("DASH"), g.get("DASH") or "not found")
if check("Grafana renders the topology panel", g.get("RENDER") == "200", f"HTTP {g.get('RENDER')}"):
    from IPython.display import Image, display
    meas.download_file(os.path.abspath("route_flow_render.png"), "/tmp/route_flow.png")
    display(Image("route_flow_render.png"))

In [ ]:
# What the meas node's route_flow service reports
info = mf.info("route_flow")
if check("mf.info('route_flow')", info.get("success"), "" if info.get("success") else info):
    print(json.dumps({k: info[k] for k in ("routes", "peers", "dashboard") if k in info}, indent=1))

## 8. ELK on the same slice
Elasticsearch and Kibana on the meas node, with Filebeat, Metricbeat and Packetbeat on every experiment node.

In [ ]:
%%time
if RUN_ELK:
    elk = (mf.instrumentize(["elk"]).get("elk") or {})
    recap = str(elk.get("play_recap", ""))
    print(recap[recap.find("PLAY RECAP"):recap.find("PLAYBOOK RECAP")])
    check("instrumentize elk", elk.get("success"), elk.get("msg", ""))

In [ ]:
%%time
def meas_json(cmd):
    out, _ = meas.execute(cmd, quiet=True)
    try:
        return json.loads(out)
    except ValueError:
        return {}

if RUN_ELK:
    es = meas_json("curl -s -m10 localhost:9200")
    es_version = es.get("version", {}).get("number")
    health = meas_json("curl -s -m10 localhost:9200/_cluster/health").get("status")
    kibana = meas_json("curl -s -m10 localhost:5601/api/status").get("version", {}).get("number")
    check("Elasticsearch up", es_version, f"version {es_version}")
    check("Elasticsearch cluster health", health in ("green", "yellow"), health)
    check("Kibana up, same version", kibana and kibana == es_version, f"Kibana {kibana}, Elasticsearch {es_version}")

    BEATS = ["filebeat", "metricbeat", "packetbeat"]
    for n in NODES:
        out, _ = slice.get_node(name=n).execute(
            "echo $(systemctl is-active " + " ".join(BEATS) + "); "
            "echo $(rpm -q --qf '%{VERSION} ' " + " ".join(BEATS) + ")", quiet=True)
        states, versions = (out.splitlines() + ["", ""])[:2]
        check(f"{n} Beats running", states.split() == ["active"] * 3, states)
        check(f"{n} Beats match Elasticsearch", versions.split() == [es_version] * 3, versions)

    # Documents from every node in every Beat (Beats 8+ report host.name in lowercase). Packetbeat can take a while.
    agg = '{"size":0,"aggs":{"h":{"terms":{"field":"host.name","size":50}}}}'
    for _ in range(12):
        hosts = {b: {x["key"] for x in meas_json(f"curl -s -m10 'localhost:9200/{b}-*/_search' "
                                                 f"-H 'Content-Type: application/json' -d '{agg}'")
                     .get("aggregations", {}).get("h", {}).get("buckets", [])} for b in BEATS}
        if all(n.lower() in hosts[b] for b in BEATS for n in NODES):
            break
        time.sleep(15)
    for b in BEATS:
        missing = [n for n in NODES if n.lower() not in hosts[b]]
        check(f"{b} documents from every node", not missing, f"missing {missing}" if missing else sorted(hosts[b]))

## 9. Summary

In [ ]:
failed = [c for c in CHECKS if not c["ok"]]
width = max(len(c["check"]) for c in CHECKS)
for c in CHECKS:
    print(f"{'PASS' if c['ok'] else 'FAIL'}  {c['check']:{width}}  {c['detail'] if not c['ok'] else ''}")
summary = {
    "result": "PASS" if not failed else "FAIL",
    "mflib_version": mflib.__version__, "mflib_git_ref": MFLIB_GIT_REF,
    "mf_ref": MF_REF, "mf_tag": MF_TAG, "mf_commit_on_meas_node": MF_COMMIT,
    "release_info": RELEASE_INFO, "slice": SLICE_NAME,
    "sites": {n: EXP[n]["site"] for n in NODES}, "meas_site": meas.get_site(),
    "started_utc": STARTED.isoformat(timespec="seconds"),
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "checks": CHECKS,
}
out_file = f"release_test_results-{mflib.__version__}-{MF_TAG}.json"
json.dump(summary, open(out_file, "w"), indent=1)
print(f"\n{summary['result']}: {len(CHECKS) - len(failed)}/{len(CHECKS)} checks passed. Saved {out_file}")

## 10. Clean up

In [ ]:
for n in NODES:
    slice.get_node(name=n).execute("pkill -f '[m]f_iperf_loop'; pkill iperf3; true", quiet=True)
print("iperf3 traffic stopped")
if DELETE_SLICE_AT_END:
    slice.delete()
    print("slice deleted")